# Week 13 Student Lab Scaffold — LLM and RAG in Healthcare

This is the **student challenge version**. You will not receive the full implementation first.

Your job is to translate pseudocode into working code. If you get stuck, use the hint cells. The instructor/reference notebook contains the complete solution.

> Clinical safety note: this notebook is for education only. Do not use generated output for patient care.


## 0. Setup

Run this cell first. It installs dependencies, loads common libraries, and configures Gemini if `GOOGLE_API_KEY` is available. If no API key is found, the lab uses fallback responses.


In [1]:
# ============================================================
# Section 0: Environment Setup
# ============================================================

!pip install langchain langchain-community langchain-core langchain-text-splitters chromadb sentence-transformers google-generativeai --quiet

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

USE_GEMINI = False
llm = None

try:
    import google.generativeai as genai
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception:
        API_KEY = os.environ.get("GOOGLE_API_KEY", "")

    if API_KEY:
        genai.configure(api_key=API_KEY)
        llm = genai.GenerativeModel("gemini-1.5-flash")
        USE_GEMINI = True
        print("Gemini API configured.")
    else:
        print("No API key found. Using fallback mode.")
except Exception as e:
    print(f"Gemini unavailable: {e}. Using fallback mode.")

print(f"Runtime mode: {'Gemini' if USE_GEMINI else 'Fallback'}")


pyenv: pip: command not found



The `pip' command exists in these Python versions:
  3.10.18

Note: See 'pyenv help global' for tips on allowing both
      python2 and python3 to be found.


No API key found. Using fallback mode.
Runtime mode: Fallback


## 1. Helper Function

This helper lets the notebook run even without a live LLM API.


In [2]:
def _fallback_response(prompt):
    prompt_lower = prompt.lower()
    if "insufficient" in prompt_lower or "not covered" in prompt_lower:
        return "Insufficient evidence in the retrieved documents. The available context does not support a safe clinical answer."
    if "reference" in prompt_lower or "cite" in prompt_lower:
        return (
            "Example response for teaching: Ceftriaxone is commonly used for suspected meningococcemia. "
            "However, any cited references must be manually verified. This fallback intentionally does not provide real citations."
        )
    if "fever" in prompt_lower and "rash" in prompt_lower:
        return (
            "Structured fallback answer:\n"
            "Key findings: fever, petechial rash, hypotension, mild neck stiffness.\n"
            "Top urgent differential: meningococcemia / meningitis with sepsis.\n"
            "Immediate actions: sepsis protocol, blood cultures, empiric antibiotics, urgent senior review.\n"
            "Missing information: exposure history, immunization status, labs, lactate, platelets, coagulation profile."
        )
    return "Fallback response: build a structured answer and state uncertainty when evidence is missing."


def query_llm(prompt, max_tokens=1024):
    if USE_GEMINI and llm is not None:
        try:
            response = llm.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(max_output_tokens=max_tokens, temperature=0.3),
            )
            return response.text
        except Exception as e:
            print(f"Gemini error: {e}. Using fallback.")
            return _fallback_response(prompt)
    return _fallback_response(prompt)


## 2. The 3 AM Clinical Case

Read the case. Before writing code, write your own top 3 differentials in a text cell or on paper.


In [3]:
clinical_case = """
Patient: 34-year-old male
Presenting at: 3 AM, Emergency Department
Chief complaint: Fever, abdominal pain, rash

Vitals:
- Temperature: 38.9 C
- Heart rate: 112 bpm
- Blood pressure: 95/60 mmHg
- Respiratory rate: 22/min
- SpO2: 96% on room air

Physical exam:
- Diffuse, non-blanching petechial rash on trunk and extremities
- Diffuse abdominal tenderness, no rebound
- Mild neck stiffness
- Alert but appears toxic

History:
- No recent travel
- No known drug allergies
- No significant past medical history
- No recent antibiotics
"""

print(clinical_case)



Patient: 34-year-old male
Presenting at: 3 AM, Emergency Department
Chief complaint: Fever, abdominal pain, rash

Vitals:
- Temperature: 38.9 C
- Heart rate: 112 bpm
- Blood pressure: 95/60 mmHg
- Respiratory rate: 22/min
- SpO2: 96% on room air

Physical exam:
- Diffuse, non-blanching petechial rash on trunk and extremities
- Diffuse abdominal tenderness, no rebound
- Mild neck stiffness
- Alert but appears toxic

History:
- No recent travel
- No known drug allergies
- No significant past medical history
- No recent antibiotics



# Lab 1 — Direct LLM + Structured Clinical Prompting

## Challenge 1A: Build the prompt yourself

Pseudocode:

```text
DEFINE clinical_case
BUILD a structured clinical prompt asking for:
  - key findings
  - top 5 differential diagnoses ranked by urgency
  - immediate actions
  - key labs
  - uncertainty / missing information
CALL query_llm(prompt)
PRINT response
```

Do not look at the reference notebook yet.


In [4]:
# TODO: Build your structured clinical prompt.
# Requirement: include key findings, differential, immediate actions, labs, and uncertainty.

prompt_structured = f"""
YOUR PROMPT HERE

Clinical case:
{clinical_case}
"""

response_structured = query_llm(prompt_structured)
print(response_structured)


Structured fallback answer:
Key findings: fever, petechial rash, hypotension, mild neck stiffness.
Top urgent differential: meningococcemia / meningitis with sepsis.
Immediate actions: sepsis protocol, blood cultures, empiric antibiotics, urgent senior review.
Missing information: exposure history, immunization status, labs, lactate, platelets, coagulation profile.


### Hint 1 — Response Schema

Ask the model to use this format:

```text
1. Key findings
2. Most urgent diagnoses, ranked
3. Immediate actions in the first 10 minutes
4. Labs and tests
5. Missing information / uncertainty
6. Safety warning
```


## Challenge 1B: Citation Stress Test

Pseudocode:

```text
BUILD a prompt asking for treatment recommendation + specific citations
FOR each citation in response:
  CHECK if citation exists
  CHECK if citation supports the exact claim
CREATE a verification table
REWRITE the answer in a safer form
```


In [5]:
# TODO: Ask for references, then verify them manually.
# Your output should include a verification table with claim / citation / exists? / supports claim?

prompt_citation_test = f"""
YOUR CITATION STRESS-TEST PROMPT HERE

Clinical case:
{clinical_case}
"""

response_citation = query_llm(prompt_citation_test, max_tokens=1500)
print(response_citation)

# TODO: Fill this table after manual verification.
verification_table = pd.DataFrame([
    {"claim": "", "citation": "", "exists": "", "supports_claim": "", "notes": ""},
])
verification_table


Structured fallback answer:
Key findings: fever, petechial rash, hypotension, mild neck stiffness.
Top urgent differential: meningococcemia / meningitis with sepsis.
Immediate actions: sepsis protocol, blood cultures, empiric antibiotics, urgent senior review.
Missing information: exposure history, immunization status, labs, lactate, platelets, coagulation profile.


,claim,citation,exists,supports_claim,notes
0,,,,,


# Lab 2 — Build a Clinical RAG System

You will now build the system that prevents the model from relying only on memory.

## Challenge 2A: Chunk the guideline corpus

Pseudocode:

```text
LOAD guideline_texts
JOIN into all_text
SPLIT all_text into overlapping chunks
PRINT number of chunks
PLOT chunk length distribution
```


In [6]:
clinical_guidelines = [
    """SEPSIS MANAGEMENT SUMMARY
    Sepsis is life-threatening organ dysfunction caused by dysregulated host response to infection.
    Obtain blood cultures before antibiotics when feasible. Measure serum lactate.
    Administer broad-spectrum IV antibiotics rapidly after sepsis recognition.
    Fluid resuscitation and vasopressors may be needed for shock.
    """,
    """MENINGOCOCCEMIA SUMMARY
    Suspected meningococcemia is a medical emergency. Fever, petechial rash, hypotension,
    and toxic appearance should trigger immediate evaluation and empiric therapy.
    Ceftriaxone or cefotaxime are commonly used empiric options, adjusted for local guidance.
    Public health notification and prophylaxis for close contacts may be required.
    """,
    """BACTERIAL MENINGITIS SUMMARY
    Bacterial meningitis may present with fever, neck stiffness, altered mental status, and sepsis.
    Blood cultures should be obtained promptly. Do not delay empiric antibiotics for imaging if unstable.
    Dexamethasone may be considered before or with the first antibiotic dose depending on context.
    """,
    """HEALTH DATA PRIVACY SUMMARY
    Clinical AI systems processing PHI require privacy safeguards. Common controls include encryption,
    access control, audit logs, de-identification when appropriate, and contractual safeguards for vendors.
    """,
]

# TODO: import the splitter, join the text, split into chunks, and inspect chunk lengths.

# from langchain_text_splitters import RecursiveCharacterTextSplitter

all_text = "\n\n".join(clinical_guidelines)

# TODO: create splitter
# splitter = ...
# chunks = ...

# print(f"Total chunks: {len(chunks)}")
# print(chunks[0][:300])


### Hint 2A — Function Names

You probably want:

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
)
chunks = splitter.split_text(all_text)
```


## Challenge 2B: Build embeddings and vector store

Pseudocode:

```text
WRAP chunks as Document objects
LOAD embedding model
BUILD Chroma vector store
CREATE retriever with k=5
TEST one similarity search
```


In [7]:
# TODO: Build the vector store.

# from langchain_community.embeddings import HuggingFaceEmbeddings
# from langchain_community.vectorstores import Chroma
# from langchain_core.documents import Document

# documents = ...
# embeddings = ...
# vectorstore = ...
# retriever = ...

# test_query = "What should we do for suspected meningococcemia?"
# retrieved_docs = retriever.invoke(test_query)
# print(retrieved_docs[0].page_content[:500])


### Hint 2B — Object Constructors

```python
documents = [Document(page_content=c, metadata={"chunk_id": i}) for i, c in enumerate(chunks)]
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={"device": "cpu"})
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings, collection_name="student_clinical_rag")
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
```


## Challenge 2C: Write the RAG functions

Pseudocode:

```text
FUNCTION retrieve_context(question):
  retrieved_docs = retriever.invoke(question)
  context = join docs as [Passage 1], [Passage 2]...
  return context, retrieved_docs

FUNCTION rag_answer(question):
  context, docs = retrieve_context(question)
  prompt = instructions + context + question
  answer = query_llm(prompt)
  return answer, docs
```


In [8]:
# TODO: Implement retrieve_context and rag_answer.

def retrieve_context(question):
    # retrieved_docs = ...
    # context = ...
    # return context, retrieved_docs
    raise NotImplementedError("Implement retrieve_context")


def rag_answer(question):
    # context, docs = retrieve_context(question)
    # prompt = f"""Use ONLY the context below..."""
    # answer = query_llm(prompt)
    # return {"answer": answer, "docs": docs, "context": context}
    raise NotImplementedError("Implement rag_answer")


## Challenge 2D: Test covered vs not-covered questions

Run one question that should be covered by the corpus and one question that is not covered.

Your RAG system should answer the covered question and say insufficient evidence for the not-covered question.


In [9]:
# TODO: Test your RAG system.

covered_question = "What immediate actions are recommended for suspected meningococcemia?"
not_covered_question = "What is the recommended HbA1c target for elderly diabetes patients with multiple comorbidities?"

# covered_result = rag_answer(covered_question)
# print(covered_result["answer"])
# print("\nRetrieved context preview:\n", covered_result["context"][:800])

# not_covered_result = rag_answer(not_covered_question)
# print(not_covered_result["answer"])


# Optional Mini-Lab — MedGemma

MedGemma is Google’s open medical model family built on Gemma. In this mini-lab, you first define the safety boundary, then optionally run inference if your environment supports it.

## Challenge: From model card to safe prototype

Pseudocode:

```text
READ the MedGemma model card
IDENTIFY:
  - model variant
  - input modality
  - output modality
  - intended use boundary
  - required validation

IF GPU + Hugging Face access are available:
  LOAD google/medgemma-1.5-4b-it
  RUN one non-PHI medical image/text example
  LABEL output as preliminary description, not diagnosis
ELSE:
  WRITE deployment pseudocode and validation plan
```


In [10]:
# TODO: Fill this table after reading the MedGemma model card.
medgemma_review = pd.DataFrame([
    {"question": "Which MedGemma variant would you use?", "your_answer": ""},
    {"question": "What input modality does your use case need?", "your_answer": ""},
    {"question": "What is the output allowed to claim?", "your_answer": ""},
    {"question": "What must be validated before clinical deployment?", "your_answer": ""},
    {"question": "Where does RAG still help?", "your_answer": ""},
])
medgemma_review


,question,your_answer
0,Which MedGemma variant would you use?,
1,What input modality does your use case need?,
2,What is the output allowed to claim?,
3,What must be validated before clinical deploym...,
4,Where does RAG still help?,


### Optional inference scaffold

Keep `RUN_MEDGEMMA = False` unless your instructor confirms that GPU and model access are available. Some MedGemma models may require accepting terms on Hugging Face.


In [11]:
RUN_MEDGEMMA = False

if RUN_MEDGEMMA:
    # TODO: Install and import the needed libraries.
    # !pip install transformers accelerate pillow --quiet
    # from transformers import pipeline
    # import torch

    # TODO: Load the model. You may need Hugging Face access approval.
    # pipe = pipeline(
    #     "image-text-to-text",
    #     model="google/medgemma-1.5-4b-it",
    #     torch_dtype=torch.bfloat16,
    #     device_map="auto",
    # )

    # TODO: Use only public, non-PHI images or text.
    # messages = [{"role": "user", "content": [
    #     {"type": "text", "text": "Describe the visible medical findings. Do not provide a final diagnosis."},
    #     {"type": "image", "url": "YOUR_PUBLIC_IMAGE_URL"},
    # ]}]
    # output = pipe(text=messages, max_new_tokens=256)
    # print(output)
    pass
else:
    print("MedGemma inference skipped. Complete the model-card review and deployment pseudocode instead.")


MedGemma inference skipped. Complete the model-card review and deployment pseudocode instead.


## Final Reflection

Answer these in prose:

1. Which part of RAG was easiest to build?
2. Which part is most likely to fail clinically?
3. What changed when you inspected retrieved passages before reading the generated answer?
4. What governance would a hospital need before using this with PHI?
